# Problem: Implement parameter initialization strategies for a CNN model in Pytorch

### Problem Statement
You are tasked with employing and evaluating a CNN model\'s parameter initialization strategies in Pytorch. 
Your goal is to initialize the weights and biases of a vanilla CNN model provided in the problem statement and comment on the implications of each strategy.

### Requirements
1. **Initialize** the weights and biases of every `Conv2d` and `Linear` layer in the following ways:
   - **Zero Initialization**: set all weights and biases to zero
   - **Random Initialization**: draw weights and biases from a standard normal distribution, N(0, 1)
   - **Xavier (Glorot) Initialization**: weights from a normal distribution with **mean=0 and variance=2 / (fan_in + fan_out)** (that is 1/n when fan_in = fan_out = n); biases zero
   - **Kaiming (He) Initialization**: weights from a normal distribution with **mean=0 and variance=2 / fan_in**, where the factor 2 compensates for ReLU zeroing half of its inputs (using fan_out instead preserves the gradient variance in the backward pass; He et al. show either works); biases zero

   Here `fan_in` is `in_features` for a `Linear` layer and `in_channels * kernel_h * kernel_w` for a `Conv2d`; `fan_out` uses `out_features` / `out_channels` instead.
2. Train and compute accuracy for each strategy
### Constraints
- Use the given CNN model and the training and testing helper functions for accuracy computations.
- Ensure the model is compatible with the CIFAR-10 dataset, which contains 10 classes.


<details>
  <summary>💡 Hint</summary>
  - Use `torch.nn.init` for weight initialization
  <br>
  - Resources to read: [All you need is a good init](https://arxiv.org/pdf/1511.06422)
</details>

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

train_dataset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=32, shuffle=True)

test_dataset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=32, shuffle=False)


In [ ]:
def train_test_loop(model, train_loader, test_loader, epochs=10):
    model.train()
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    for epoch in range(epochs):
        for image, label in train_loader:
            pred = model(image)
            loss = criterion(pred, label)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()    
        print(f"Training loss at epoch {epoch} = {loss.item()}")
    
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for image_test, label_test in test_loader:
            pred_test = model(image_test)
            _, pred_test_vals = torch.max(pred_test, dim=1)
            total += label_test.size(0)
            correct += (pred_test_vals == label_test).sum().item()
    print(f"Test Accuracy = {(correct * 100)/total}")

In [ ]:
class VanillaCNNModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.fc1 = nn.Linear(64*16*16, 128)
        self.fc2 = nn.Linear(128, 10)
        self.relu = nn.ReLU()
    
    def forward(self, x):
        x = self.relu(self.conv1(x))
        x = self.pool(self.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

In [ ]:
def config_init(init_type="kaiming"):
    """Return an initializer for `model.apply(...)`. It only touches Conv2d and Linear
    layers, because apply() also visits the container, ReLU and MaxPool modules."""

    def kaiming_init(m):
        # He et al. (2015): Var(W) = 2 / fan_in keeps the activation variance constant
        # through ReLU layers. (mode='fan_out' preserves the gradient variance instead.)
        if isinstance(m, (nn.Conv2d, nn.Linear)):
            nn.init.kaiming_normal_(m.weight, mode='fan_in', nonlinearity='relu')
            if m.bias is not None:
                nn.init.zeros_(m.bias)

    def xavier_init(m):
        # Glorot & Bengio (2010): Var(W) = 2 / (fan_in + fan_out)
        if isinstance(m, (nn.Conv2d, nn.Linear)):
            nn.init.xavier_normal_(m.weight)
            if m.bias is not None:
                nn.init.zeros_(m.bias)

    def zeros_init(m):
        if isinstance(m, (nn.Conv2d, nn.Linear)):
            nn.init.zeros_(m.weight)
            if m.bias is not None:
                nn.init.zeros_(m.bias)

    def random_init(m):
        # N(0, 1), deliberately ignoring the layer's fan-in
        if isinstance(m, (nn.Conv2d, nn.Linear)):
            nn.init.normal_(m.weight)
            if m.bias is not None:
                nn.init.normal_(m.bias)

    initializer_dict = {"kaiming": kaiming_init,
                        "xavier": xavier_init,
                        "zeros": zeros_init,
                        "random": random_init}

    return initializer_dict.get(init_type)

In [ ]:

for name, model in zip(["Vanilla", "Kaiming", "Xavier", "Zeros", "Random"], [VanillaCNNModel(),
              VanillaCNNModel().apply(config_init("kaiming")),
              VanillaCNNModel().apply(config_init("xavier")),
              VanillaCNNModel().apply(config_init("zeros")),
              VanillaCNNModel().apply(config_init("random"))
              ]):
    print(f"_________{name}_______________________")
    train_test_loop(model, train_loader, test_loader)
    

In [ ]:
# Auto-grade (one-time setup from the repo root: pip install -e python/)
from torchleet import check
check("cnn-param-init", config_init=config_init)

### What to expect, and why

- **Zeros**: every unit in a layer computes the same thing, gets the same gradient and so stays identical — symmetry is never broken. With ReLU it is worse still: every activation is 0, so the only parameter that receives any gradient is the last layer's bias. The network can only learn the class prior and stays at chance (~10% on CIFAR-10).
- **Random N(0, 1)**: a layer's output variance is about `fan_in * Var(W) * E[x^2]`, and here `fan_in` ranges from 27 to 16384, so activations grow by up to ~100x per layer. The logits are enormous, the softmax saturates, the initial loss is in the tens of thousands and training is slow and unstable.
- **Xavier**: derived for linear / tanh units, it keeps the variance of the activations (forward) and of the gradients (backward) roughly constant — 2 / (fan_in + fan_out) is the compromise between the two. ReLU zeroes half of its inputs, so each layer halves the variance: harmless for this 4-layer net, a real problem for deep ones (`nn.init.xavier_normal_(w, gain=nn.init.calculate_gain('relu'))` fixes it).
- **Kaiming**: variance 2 / fan_in exactly compensates for ReLU's halving, so the signal neither explodes nor vanishes with depth — the standard choice for ReLU networks.
- **Vanilla (PyTorch default)**: `nn.Conv2d` and `nn.Linear` use `kaiming_uniform_(a=sqrt(5))`, i.e. U(-1/sqrt(fan_in), 1/sqrt(fan_in)), so Var(W) = 1 / (3 * fan_in), with the bias drawn from the same range.

On a network this shallow, trained with Adam, Xavier, Kaiming and the default all train well, zero init is stuck at chance and N(0, 1) is clearly worse. The differences between the sensible schemes grow with depth and with plain SGD.